# SAE steering analysis — do our SAEs learn meaningful, *causal* features?

This notebook proves our trained BatchTopK SAEs learn meaningful features, and that the
**FP8-trained SAE is causally equivalent to the FP16 baseline**, using two visualizations:

1. **Token-activation heatmaps** (Anthropic *Scaling Monosemanticity* style): color each
   token by how strongly a feature fires on it — shows *what* the feature detects.
2. **Steering generations**: clamp a feature to `±x × (its max activation in the dataset)`
   along its decoder direction and read off how the model's output changes. `x` is a
   parameter you sweep; `+x` amplifies the concept, `-x` suppresses it.

We pick **the same concept feature independently in each SAE** (FP8 and FP16 have unrelated
feature indices) by matching each feature's top-activating-token class to a keyword, then
compare their heatmaps and steering effects side by side.

> Prereq: run `experiments/steering_find_features.py` on both SAEs first (it writes the
> `steering/concrete_features.{json,npz}` this notebook reads). Already done for
> `results/saebench_gemma{,_fp8}/w65536_k80`.

In [1]:
import sys
from pathlib import Path

import torch
from IPython.display import HTML, display

# Make experiments/steering_lib.py importable.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "experiments"))
import steering_lib as S

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)
print("device:", device)

device: cuda


## Configuration

Edit these to point at your two SAEs, pick the concept, and set the steering sweep `x`
(applied as both `+x` and `-x`). `CONCEPTS` maps a search keyword → a few target tokens
used for the quantitative Δ log-prob curve at the end.

In [2]:
# The two SAEs to compare (same recipe, different GEMM precision). Add more if you like.
SAES = {
    "FP16": REPO / "experiments/results/saebench_gemma/w65536_k80",
    "FP8":  REPO / "experiments/results/saebench_gemma_fp8/w65536_k80",
}
DTYPE = "bfloat16"

# Concepts to analyze. Each maps a search keyword (used to FIND the feature in each SAE)
# -> target tokens for the Δ log-prob curve. CONCEPTS_TO_RUN picks which to run.
CONCEPTS = {
    "mountain":   [" mountain", " mountains", " Mountain", " Mount"],
    "cell":       [" cell", " cells", " cellular"],
    "patient":    [" patient", " patients"],
    "soybean":    [" soybean", " alfalfa", " quinoa"],
    "regression": [" regression", " model", " analysis"],
    "value":      [" value", " values"],
}
CONCEPTS_TO_RUN = ["mountain", "cell", "patient", "soybean", "regression"]

# Heatmaps: number of REAL max-activating examples to show per concept, and the token
# budget for finding them (rarer features need a bigger scan to fill all examples).
N_HEATMAP_EXAMPLES = 5
EXAMPLES_SCAN_TOKENS = 250_000

# Steering: for each concept we CLAMP the feature's activation to (x × its dataset-max
# activation) for each x below, generating N_GEN continuations per x. x=0 is the genuine
# unsteered baseline; +x amplifies the concept, -x suppresses it. Same x set used for the
# Stage D curves.
STEER_X_SWEEP = [-5, -2, 0, 2, 5]
N_GEN = 3

# A neutral-but-fitting seed message per concept (a lead-in where the concept could
# plausibly surface, so baseline text stays generic and steering effects are obvious).
CONCEPT_PROMPTS = {
    "mountain":   "Last weekend I decided to go on a trip and",
    "cell":       "In science class today we were studying the human body and",
    "patient":    "When I walked into the hospital that morning,",
    "soybean":    "The farmer looked out over his fields and",
    "regression": "To analyze the dataset, the researchers",
}
DEFAULT_STEER_PROMPT = "I went for a walk and"

GEN_KWARGS = dict(max_new_tokens=40, temperature=1.0, top_p=0.3, freq_penalty=1.0, seed=0)

In [3]:
# Load each SAE (the base gemma model is read from the SAE cfg and cached, so both SAEs
# share one copy of the LLM in memory).
saes = {}
model = None
for name, sae_dir in SAES.items():
    model, sae, meta = S.load_model_and_sae(sae_dir, device=device, dtype=DTYPE)
    saes[name] = dict(dir=sae_dir, sae=sae, meta=meta)
    print(f"{name:5s}  {sae_dir.name}  hook={meta.hook_name}  d_sae={sae.cfg.d_sae}")

/opt/venv/lib/python3.12/site-packages/apex/transformer/functional/fused_rope.py:54: UserWarning: Using the native apex kernel for RoPE.
  warnings.warn("Using the native apex kernel for RoPE.", UserWarning)
/opt/venv/lib/python3.12/site-packages/sae_lens/saes/sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
FP16   w65536_k80  hook=blocks.12.hook_resid_post  d_sae=65536
FP8    w65536_k80  hook=blocks.12.hook_resid_post  d_sae=65536


## Stage A — find each concept's feature in each SAE + collect its examples

FP8 and FP16 have unrelated feature indices, so for every concept we locate the feature
*independently* in each SAE by matching its top-activating-token class to the keyword. We
then stream the dataset once per SAE to collect each chosen feature's real max-activating
examples (used for the Stage B heatmaps).

In [4]:
# Find the best-matching feature for each concept INDEPENDENTLY in each SAE.
concepts = {}  # concept -> {SAE -> {feature, max_act, token_class, examples}}
for c in CONCEPTS_TO_RUN:
    concepts[c] = {}
    print(f"\n=== concept '{c}' ===")
    for name, d in saes.items():
        hits = S.find_feature_by_keyword(d["dir"], c, top_k=5)
        if not hits:
            print(f"  {name:5s} no feature matched '{c}' — skipped in this SAE")
            continue
        h = hits[0]
        concepts[c][name] = {"feature": h["feature"],
                             "max_act": S.feature_max_act(d["dir"], h["feature"]),
                             "token_class": h["token_class"]}
        print(f"  {name:5s} feat {h['feature']:>6}  max={h['max_activation']:>6.1f}  "
              f"class={h['token_class']}")

# Collect each feature's real max-activating examples — ONE dataset pass per SAE for all
# concepts (the authentic Anthropic-style feature visualization used in Stage B).
for name, d in saes.items():
    feats = [concepts[c][name]["feature"] for c in CONCEPTS_TO_RUN if name in concepts[c]]
    print(f"\nCollecting max-activating examples for {name} "
          f"({len(feats)} features, {EXAMPLES_SCAN_TOKENS:,} tokens)...")
    ex = S.top_activating_examples(model, d["sae"], feats,
                                   n_examples=N_HEATMAP_EXAMPLES,
                                   n_tokens=EXAMPLES_SCAN_TOKENS)
    for c in CONCEPTS_TO_RUN:
        if name in concepts[c]:
            concepts[c][name]["examples"] = ex[concepts[c][name]["feature"]]
print("done.")


=== concept 'mountain' ===
  FP16  feat   3530  max=  39.5  class=[' Alps', ' the', ' ranges', ' are', 'Mountain', ' Pass']
  FP8   feat  49384  max=  50.2  class=[' Mt', 'Mt', 'Mountain', ' mountains', 'ountain', ' Mountain']

=== concept 'cell' ===
  FP16  feat  20318  max=  43.5  class=[' cell', ' cells', ' Cell', 'Cell', ' Cells']
  FP8   feat   5219  max=  46.8  class=[' cell', ' cells', ' Cell', 'Cell', ' Cells']

=== concept 'patient' ===
  FP16  feat   2139  max=  49.8  class=[' patients', ' patient', ' Patients', 'Patient', 'Patients']
  FP8   feat  37031  max=  51.5  class=[' patients', ' patient', ' Patients', 'Patient', 'Patients']

=== concept 'soybean' ===
  FP16  feat  54048  max=  48.5  class=[' isof', ' soybean', 'lav', ' Soybean', 'soy', 'bean']
  FP8   feat  21880  max=  47.8  class=[' isof', ' soybean', 'bean', ' Soybean']

=== concept 'regression' ===
  FP16  feat  56873  max=  29.8  class=[' regression', ' model', ' analysis', ' fit', ' run', ' an']
  FP8   feat 

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

done.


## Stage B — token-activation heatmaps (what each feature detects)

For every concept (5), we show the feature's `N_HEATMAP_EXAMPLES` (5) strongest real
contexts from the dataset, with each token colored by the feature's activation (normalized
by the feature's dataset-max, so FP8 and FP16 share a color scale). A meaningful,
monosemantic feature lights up on the concept tokens and stays dark elsewhere — and the FP8
and FP16 panels should look essentially identical.

In [5]:
for c in CONCEPTS_TO_RUN:
    display(HTML(f"<h3>Concept: <code>{c}</code> — top {N_HEATMAP_EXAMPLES} "
                 f"max-activating examples</h3>"))
    cols = []
    for name in saes:
        if name not in concepts[c]:
            continue
        info = concepts[c][name]
        blocks = [f"<div style='font:13px monospace;margin:4px 0'><b>{name}</b> · "
                  f"feature {info['feature']} · class {info['token_class'][:4]}</div>"]
        for e in info.get("examples", []):
            blocks.append(S.token_heatmap_html(e["str_tokens"], e["acts"],
                                               max_act=info["max_act"]))
        if not info.get("examples"):
            blocks.append("<div style='color:#a00'>no examples found — raise "
                          "EXAMPLES_SCAN_TOKENS</div>")
        cols.append("<div style='flex:1;min-width:400px;margin:4px'>"
                    + "".join(blocks) + "</div>")
    display(HTML("<div style='display:flex;flex-wrap:wrap'>" + "".join(cols) + "</div>"))

## Stage C — steering generations across an x sweep per concept

For each concept we **clamp** the feature's activation to `x × max_act` for `x` in
`STEER_X_SWEEP` (`[-5, -2, 0, +2, +5]`), generating `N_GEN` (3) continuations at each `x`.
`x=0` is the genuine unsteered baseline. Reading down a column, `+x` should pull the text
toward the concept and `-x` away from it — the same way for FP8 and FP16.

In [ ]:
import html as _html

def gen_block(text_list):
    return "".join(f"<div style='margin:3px 0;padding:4px;border-left:3px solid #ccc'>"
                   f"{_html.escape(t.replace(chr(10), ' '))}</div>" for t in text_list)

for c in CONCEPTS_TO_RUN:
    prompt = CONCEPT_PROMPTS.get(c, DEFAULT_STEER_PROMPT)
    display(HTML(f"<h3>Concept: <code>{c}</code> — clamp to x×max_act for x in "
                 f"{STEER_X_SWEEP}, {N_GEN} generations each</h3>"
                 f"<div style='color:#888;font:12px monospace'>prompt: "
                 f"{_html.escape(prompt)}</div>"))
    rows = []
    for x in STEER_X_SWEEP:
        label = "baseline (x=0)" if x == 0 else f"x={x:+g}× max_act"
        bg = "#eef" if x == 0 else ("#e9f7e9" if x > 0 else "#fdeaea")
        cells = []
        for name in saes:
            if name not in concepts[c]:
                cells.append("<td>—</td>")
                continue
            info = concepts[c][name]
            outs = S.steer_generate(model, saes[name]["sae"], info["feature"],
                                    prompt, x, info["max_act"],
                                    n_samples=N_GEN, **GEN_KWARGS)
            cells.append("<td style='padding:6px;border:1px solid #eee;vertical-align:top;"
                         f"font:12px monospace;max-width:520px'>{gen_block(outs)}</td>")
        rows.append(f"<tr><td style='background:{bg};padding:6px;font-weight:bold'>"
                    f"{label}</td>" + "".join(cells) + "</tr>")
    header = "<th></th>" + "".join(
        f"<th style='padding:6px'>{n} · feat {concepts[c][n]['feature']}</th>"
        for n in saes if n in concepts[c])
    display(HTML(f"<table style='border-collapse:collapse'><tr>{header}</tr>"
                 + "".join(rows) + "</table>"))

## Stage D — quantitative steering curves (FP8 vs FP16), one per concept

A non-anecdotal readout: the mean next-token log-prob of each concept's target tokens vs
steering strength `x` (swept over `STEER_X_SWEEP`). If FP8 learned the same causal feature
as FP16, the two curves track each other.

In [ ]:
import numpy as np
import plotly.graph_objects as go

signed_xs = sorted(STEER_X_SWEEP)
for c in CONCEPTS_TO_RUN:
    prompt = CONCEPT_PROMPTS.get(c, DEFAULT_STEER_PROMPT)
    fig = go.Figure()
    for name in saes:
        if name not in concepts[c]:
            continue
        info = concepts[c][name]
        ys = [float(S.target_logprob(model, saes[name]["sae"], info["feature"],
                                     prompt, CONCEPTS[c], x, info["max_act"]))
              for x in signed_xs]
        fig.add_trace(go.Scatter(x=signed_xs, y=ys, mode="lines+markers", name=name))
    fig.update_layout(
        title=f"'{c}' — steering effect on {CONCEPTS[c]}",
        xaxis_title="steering x  (feature clamped to x × dataset-max)",
        yaxis_title="mean log-prob of target tokens", width=720, height=420,
        template="plotly_white",
    )
    fig.show()

## Takeaways

- **Heatmaps (Stage B)** show the feature fires specifically on the concept tokens →
  the SAE learned a meaningful, monosemantic feature, not noise.
- **Steering (Stage C)** shows clamping that single feature *causally* steers generation
  toward (`+x`) / away from (`-x`) the concept → the feature is behaviorally real.
- **Curves (Stage D)** show the FP8 and FP16 features produce the **same steering response**
  → FP8 training preserves the causal feature geometry; it's not just close on
  reconstruction metrics, it's interpretably and causally equivalent.

To change which concepts run, edit `CONCEPTS` / `CONCEPTS_TO_RUN` in the config. To swap
models/SAEs, edit `SAES` — everything else (base model, hook, layer) is read from each SAE's
own config.